In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np

# Read all pdfs, inside the directory 

In [ ]:
def process_all_documents(pdf_dir):
    all_documents = []
    pdf_directory = Path(pdf_dir)
    
    pdf_files = list(pdf_directory.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf file to process")
    
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} page")
            
        except Exception as e:
            print(f"Error: {e}")
        
    print(f"Total loaded documents: {len(all_documents)}")
    return all_documents

In [ ]:
dir = "../data/pdf_files"
all_documents = process_all_documents(dir)

In [ ]:
all_documents

# Chunking 

In [ ]:
def split_documents(documents, chunk_size = 1000, overlap_size = 100) -> np.array:
    """
    function for spliting documents for better optimization for a RAG system
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap_size,
        length_function = len,
        separators=['\n\n', '\n', ' ', '']
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"\t1. Content: {split_docs[0].page_content[:100]}")
        print(f"\t2. Metadata: {split_docs[0].metadata}")
    
    return split_docs

docs_chunked = split_documents(all_documents, chunk_size=1000, overlap_size=100)

# Embeddings 

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbManagaer():
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        """
        Initialize the embeddings manager
        
        Args:
            - model_name: HuggingFace model name for sentence embeddings    
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """
        Load The SentenceTransformer model
        """
        try:
            print(f'Loading the model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sccessfully, emb dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 
        
        
    def generate_embeddings(self, texts):
        """
        Generate embeddings ofr a list of texts
        
        Args:
            - texts: List of string texts

        Returns:
            - numpy array of embeddings with shape (len(texts), embeddings_dim)
        """
        
        if not self.model:
            raise ValueError('Model not Loaded')
        
        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        
        return embeddings


# init the embeddings manager
emb_manager = EmbManagaer("all-MiniLM-L6-v2")
emb_manager

# Vector DB

In [ ]:
class VectorStore:
    """
    Manages the vector store in a ChromeDB vector db
    """
    def __init__(self, collection_name, persist_directory):
        """
        Init the vector store

        Args: 
            - collection_name: Name of Chroma collection
            - persist_directory; Directory in which we persist the db
        """
    
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """
        Initialize ChromaDB Client and Collection
        """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }    
            )
            
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error intializing vector store: {e}")
            raise
    
    def add_document(self, documents, embeddings):
        """
        Add documents and thier embeddings to the vector store
        
        Args:
            - documents: List of langchain document
            - embeddings: Correspoding embeddings for those documents
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        
        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embeddings 
            embeddings_list.append(embeddings.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            
            print(f"Successfully added {len(documents)} document to the DB")
            print(f"Total document in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


vector_store = VectorStore(collection_name="pdfs", persist_directory="../data/vector_store")
vector_store

In [ ]:
texts = [chunk.page_content for chunk in docs_chunked]
embeddings = emb_manager.generate_embeddings(texts)
vector_store.add_document(docs_chunked, embeddings)

In [ ]:
docs_chunked

# Retriever Pipline from vector store

In [ ]:
class RAGRetriever:
    """
    Handles query-based retrieval from the vector store
    """
    
    def __init__(self, vector_store, embeddings_manager):
        """
        Initialize the retriever
        
        Args:
            - vector_store: Vector store containing document embeddings
            - embeddings_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embeddings_manager = embeddings_manager
        
    def retrieve(self, query, top_k, score_threshold):
        """
        Retrieve relevant documents for a query
        
        Args:
            - query: The search query
            - top_k: Number of top results to return
            - score_threshold: Minimum similarity score threshold
            
        Returns:
            - List of dictionaries containing retrieved documents and metadata
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")
        
        # Generate query embeddings
        query_embeddings = self.embeddings_manager.generate_embeddings([query])[0]
        
        # Search in Vector DB
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k
            )
            
            # Process results
            retrieved_docs = []
            if results["documents"] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance) 
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after filtering")
            else:
                print("No document found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error During retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, emb_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("What is kafka", 5, 0.0)

# From Vector DB to LLM output generation

### API KEY, FREE FROM GROQ

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("groq_api_key"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name="gemma2-9b-it", api_key = None):
        """
        Initialize Groq LLM
        
        Args:
            - model_name: Groq model name 
            - api_key: Groq api key
        """
        
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required.")
        
        self.llm = ChatGroq(
            model = self.model_name,
            api_key = self.api_key,
            temperature= 0.1,
            max_tokens=1024
        )
        
        print(f"Initialized GROQ LLM with model: {self.model_name}")
        
    
    def generate_response(self, query, context, max_length=500):
        """
        Generate Response using retrieved context
        
        Args:
            - query: User question
            - context: Retrieved document context
            - max_length: Maximum response length
            
        Returns:
            - Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""
            You are a helpful AI assistant. Use the following context, to answer the question accurately and concisely.

            Context: {context}
            
            Question: {question}
            
            Answer: Provide a clear and informative answer based on the context above. if the context doesnt contain enough information to answer the question, please say so.
            """
        )
        
        # format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate the response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating the response: {str(e)}"
    
    def generate_response_simple(self, query, context):
        """
        Simple response generation without complex prompting
        Args:
            - query: User question
            - context: Retrieved context
        Returns:
            - Generated Response      
        """
        
        simple_prompt = f"""
        Based on this context: {context}
        Question: {query}
        Answer:
        """
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

### Init the class

In [ ]:
try:
    groq_llm = GroqLLM(api_key=os.getenv("groq_api_key"))
    print("Groq LLM initialized succesfully")

except Exception as e:
    print(f"Warining: {e}")
    print("Please set env key")
    groq_llm = None

In [ ]:
rag_retriever.retrieve('Super resolution best model in the study', top_k=5, score_threshold=0.0)

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Init Groq LLM with env variable
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

# Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieving the context
    results = retriever.retrieve(query, top_k=top_k, score_threshold=0.0)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ''
    if not context:
        return "No relevant context found to answer the question"
    
    prompt = f"""
    Use the following context to answer the question concisely.
    Context: {context}
    
    Question: {query}
    
    Answer:
    """
    
    response = llm.invoke([prompt.format(context, query=query)])
    return response.content

In [ ]:
answer = rag_simple("What are the limits of the satellite super resolution using deep leaning projet, was the variance a problem?", rag_retriever, llm)
print(answer)

In [ ]:
answer = rag_simple("how ESRGAN work ? in super resolution?", rag_retriever, llm)
print(answer)

# Enhanced RAG Pipline Features

In [ ]:
def rag_advanced(query, retriever, llm, top_k, min_score, return_context):
    """
    RAG pipline with extra features:
    returns answer, sources, confidence score, and optionally full context
    """
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and resources
    context = '\n\n'.join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unkown')),
        'page': doc['metadata'].get('page', 'unkown'),
        'score': doc['metadata'].get('similarity_score'),
        'preview': doc['content'][:300] + "..."
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""
    Use the following context to answer the question concisely.
    Context: {context}
    Question: {query}
    Answer:
    """
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    return output

In [ ]:
result = rag_advanced("how ESRGAN work ? in super resolution?", rag_retriever, llm, 5, 0.0, return_context=True)
print(f"Answer: {result['answer']} \n########\n")
print(f"Sources: {result['sources']}\n########\n")
print(f"Confidence: {result['confidence']}\n########\n")
print(f"Context: {result['context']}\n########\n")

# Advanced RAG pipline: Streaming Citations, History, Summarization

In [51]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []
    
    def query(self, question, top_k=5, min_score=0.2, stream=False, summarize=False):
        # Retriever relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ''
        else:
            context = '\n\n'.join([doc['content']  for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unkown')),
                'page': doc['metadata'].get('page', 'unkown'),
                'score': doc['similarity_score'],
                'preview': doc['content']
            } for doc in results]
            
            # Streaming answer simulation
            prompt = f"""
            Use the following context to answer the question concisely.
            Context: {context}
            Question: {question}
            Answer:
            """
            
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content
            
        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitation:\n" + '\n'.join(citations) if citations else answer
        
        # Optionally summarize the answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences: \n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        
        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }
        
        


In [52]:
adv_rag = AdvancedRAGPipline(rag_retriever, llm)
result = adv_rag.query("how ESRGAN work ? in super resolution?", 5, 0.0, False, True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'how ESRGAN work ? in super resolution?'
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 254.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents after filtering



Final Answer: ESRGAN works in two stages to ensure training stability and high perceptual quality in super-resolution:

1. **Stage 1: Generator Pre-training** - The generator is trained using a pixel-wise ℓ1 loss to learn a reliable mapping from low-resolution images to high-resolution images.
2. **Stage 2: Adversarial Training** - The generator and discriminator are trained together, with the generator trying to produce realistic high-resolution images and the discriminator trying to distinguish between real and generated images. The learning rate is reduced at epochs 15, 30, and 45 to refine high-frequency details.

This process allows ESRGAN to produce visually sharper and more detailed results, with enhanced textures, stronger edge definition, and improved visual realism, closely resembling the high-resolution ground truth.

Citation:
[1] SISR.pdf (page 21)
[2] SISR.pdf (page 30)
[3] SISR.pdf (page 16)
[4] SISR.pdf (page 13)
Summary: ESRGAN is a two-stage super-resolution process 